# latex-data · Dataset filtering & statistics → LaTeX

Computes the affinity filtering funnel, sequence-clustering tiers, and per-split statistics **fresh** from the frozen split manifests (`voxbind/splits/*.csv`) + `LP_PDBBind.csv`, plus descriptors for the CrossDocked and FuncBind-MCP generation benchmarks.

## Setup + helpers

In [ ]:
import sys, csv, statistics as st
from pathlib import Path
from collections import Counter, defaultdict
def _find_repo_root():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "voxbind" / "dataset").is_dir():
            return c
    fb = Path("/home/shpark/prj-denovo/VoxBind")
    if (fb / "voxbind" / "dataset").is_dir():
        return fb
    raise FileNotFoundError("repo root not found")
_REPO = _find_repo_root()

def latex_table(caption, label, header, rows, colspec, wide=False):
    env = "table*" if wide else "table"
    out = [f"\\begin{{{env}}}[t]", "  \\centering", f"  \\caption{{{caption}}}",
           f"  \\label{{{label}}}", f"  \\begin{{tabular}}{{{colspec}}}", "    \\toprule",
           "    " + " & ".join(header) + " \\\\", "    \\midrule"]
    for r in rows:
        out.append("    " + " & ".join(str(x) for x in r) + " \\\\")
    out += ["    \\bottomrule", "  \\end{tabular}", f"\\end{{{env}}}"]
    return "\n".join(out)

LP = list(csv.DictReader(open(_REPO/"voxbind/dataset/data/pdbbind/raw/LP_PDBBind.csv")))
LP_BY_PID = {r[""]: r for r in LP}
def manifest(name):
    return list(csv.DictReader(open(_REPO/f"voxbind/splits/{name}.csv")))
def sp_counts(rows, key="split"):
    c = Counter(r[key] for r in rows if r.get(key) in ("train","val","test"))
    return [c["train"], c["val"], c["test"], c["train"]+c["val"]+c["test"]]
print("repo:", _REPO, "| LP rows:", len(LP))


## Filtering funnel (affinity)

In [ ]:
# ── Filtering funnel (affinity dataset: lp_edrscc_v2) ──
lp_full = sp_counts(LP, "new_split")
v1 = sp_counts(manifest("lp_edrscc_v1"))
v2 = sp_counts(manifest("lp_edrscc_v2"))
rows = [
    ["LP-PDBBind (Li et al., 2021)", *lp_full],
    [r"$\cap$ electron density $\cap$ lig+poc RSCC\,$\geq$\,0.8", *v1],
    [r"$\cap$ Kd/Ki only (drop IC$_{50}$) \;=\; \textbf{lp\_edrscc\_v2}", *v2],
]
print(latex_table(
    r"\textbf{Affinity dataset filtering funnel.} Complex counts after each filter on the LP-PDBBind split.",
    "tab:data-funnel", ["Filtering stage", "Train", "Val", "Test", "Total"], rows, "lrrrr"))


## Sequence-clustering tiers

In [ ]:
# ── Sequence-clustering tiers on lp_edrscc_v2 ──
rows = []
for name, disp in [("lp_edrscc_v2","lp\\_edrscc\\_v2 (base)"),
                   ("lp_edrscc_v2_cl1","+CL1"), ("lp_edrscc_v2_cl12","+CL1+CL2"),
                   ("lp_edrscc_v2_cl123","+CL1+CL2+CL3")]:
    rows.append([disp, *sp_counts(manifest(name))])
print(latex_table(
    r"\textbf{Leakage-controlled sequence-clustering tiers.} Nested MMseqs2 identity filters removing train--test similar complexes.",
    "tab:data-cl-tiers", ["Split", "Train", "Val", "Test", "Total"], rows, "lrrrr"))


## Per-split statistics (lp_edrscc_v2)

In [ ]:
# ── Per-split statistics for lp_edrscc_v2 ──
def stat_row(pids, label):
    recs = [LP_BY_PID[p] for p in pids if p in LP_BY_PID]
    prot = len({r["seq"] for r in recs})
    kd = sum(1 for r in recs if r["kd/ki"].strip().lower().startswith("kd"))
    ki = sum(1 for r in recs if r["kd/ki"].strip().lower().startswith("ki"))
    vals = [float(r["value"]) for r in recs if r["value"] not in ("","nan")]
    res  = [float(r["resolution"]) for r in recs if r["resolution"] not in ("","nan")]
    refined = sum(1 for r in recs if r["category"]=="refined")
    general = sum(1 for r in recs if r["category"]=="general")
    return [label, len(recs), prot, f"{kd}/{ki}", f"{st.median(vals):.2f}",
            f"[{min(vals):.1f}, {max(vals):.1f}]", f"{st.median(res):.2f}", f"{refined}/{general}"]
v2m = manifest("lp_edrscc_v2")
by_split = defaultdict(list)
for r in v2m: by_split[r["split"]].append(r["pid"])
rows = [stat_row(by_split[s], s.capitalize()) for s in ("train","val","test")]
print(latex_table(
    r"\textbf{lp\_edrscc\_v2 dataset statistics.} Affinity is $-\log K_{d/i}$ (median and range over complexes). Unique proteins by sequence; Refined/General = PDBbind quality tier.",
    "tab:data-stats",
    ["Split", "Complexes", "Proteins", "Kd/Ki", "Aff. med.", "Aff. range", "Resol. (\\AA)", "Refined/Gen."],
    rows, "lrrrrrrr", wide=True))


## Generation benchmarks (CrossDocked, MCP)

In [ ]:
# ── Generation benchmarks (task2 CrossDocked, task3 MCP) — descriptors ──
rows = [
    ["De novo drug design", "CrossDocked2020 test", "100 pockets (79 w/ electron density)", "100 mol./pocket sampled"],
    ["Macrocyclic peptides", "FuncBind MCP test (pilot)", "10 targets", "25 samples/receptor, 256 chains, 1 attempt"],
]
print(latex_table(
    r"\textbf{Generation benchmarks.} Evaluation sets for the de novo drug-design (task 2) and macrocyclic-peptide (task 3) tasks.",
    "tab:data-generation", ["Task", "Benchmark", "Size", "Sampling"], rows, "llll", wide=True))
print("\n% source: results/reports/results_drug_design.html, results/reports/results_mcp.html")
